In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import os
from sklearn.metrics import classification_report, confusion_matrix
import pickle

from datasets import Dataset, DatasetDict
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer, pipeline
from transformers import BertTokenizer, BertForSequenceClassification, RobertaForSequenceClassification, RobertaTokenizer
from torchinfo import summary

BASE_PATH = os.path.dirname(os.getcwd())
DATASET_PATH = BASE_PATH+"/datasets"

with open(f"{BASE_PATH}/datasets/material/readyDataset.pkl", "rb") as f : 
    df = pickle.load(f)
print("LOADED")

LOADED


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

labelEncoder = LabelEncoder()
LABEL = labelEncoder.fit_transform(df["label"])
train_data, test_data, train_label, test_label = train_test_split(df["text"], LABEL, test_size = 0.2, random_state=42, stratify=LABEL)

### 1. DistilBERT

In [5]:
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels = 6)
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
summary(model)

Layer (type:depth-idx)                                  Param #
DistilBertForSequenceClassification                     --
├─DistilBertModel: 1-1                                  --
│    └─Embeddings: 2-1                                  --
│    │    └─Embedding: 3-1                              23,440,896
│    │    └─Embedding: 3-2                              393,216
│    │    └─LayerNorm: 3-3                              1,536
│    │    └─Dropout: 3-4                                --
│    └─Transformer: 2-2                                 --
│    │    └─ModuleList: 3-5                             42,527,232
├─Linear: 1-2                                           590,592
├─Linear: 1-3                                           4,614
├─Dropout: 1-4                                          --
Total params: 66,958,086
Trainable params: 66,958,086
Non-trainable params: 0

In [7]:
train_dataset = Dataset.from_dict({"text": train_data.tolist(),"labels" : train_label.tolist()})
test_dataset = Dataset.from_dict({"text": test_data.tolist(),"labels" : test_label.tolist()})

# Gabungkan menjadi DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [8]:
def tokenization(batch) : 
    return tokenizer(batch["text"], max_length = 100, padding = 'max_length', truncation = True, return_attention_mask = True)

dataset = dataset.map(tokenization, batch_size=32, batched = True)

Map: 100%|██████████| 4000/4000 [00:02<00:00, 1964.40 examples/s]


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def comp_metrics(acc) : 
    logits, labels = acc
    logits = np.argmax(logits, axis = -1)
    accuracy = accuracy_score(logits, labels)
    precision = precision_score(logits, labels, average='weighted')
    recall = recall_score(logits, labels, average='weighted')   
    f1 = f1_score(logits, labels, average='weighted')
    return {"f1" : f1, "precision" : precision, "recall" : recall, "accuracy" : accuracy}

trainingArgs = TrainingArguments(
    per_device_train_batch_size = 16,
    num_train_epochs = 10,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    fp16 = True,
    eval_strategy = "epoch",
    per_device_eval_batch_size = 8,
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "eval_f1",
    greater_is_better = True,
    seed = 42
)
trainer = Trainer(
    args = trainingArgs,
    model = model,
    train_dataset= dataset["train"],
    eval_dataset = dataset["test"],
    compute_metrics=comp_metrics
)

In [10]:
trainer.train()
trainer.evaluate()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.257000,0.204053,0.923351,0.924495,0.923250,0.923250
2,0.150300,0.186519,0.931076,0.931730,0.931750,0.931750
3,0.119200,0.201780,0.930719,0.934929,0.929250,0.929250
4,0.092300,0.226214,0.927462,0.927729,0.927750,0.927750
5,0.065600,0.289978,0.926185,0.926560,0.927250,0.927250
6,0.046800,0.338282,0.927496,0.927282,0.928000,0.928000
7,0.040400,0.367740,0.928908,0.928931,0.929250,0.929250
8,0.020200,0.409693,0.926428,0.926801,0.926750,0.926750
9,0.018200,0.415976,0.930347,0.930368,0.930500,0.930500
10,0.016300,0.419188,0.928607,0.928479,0.929000,0.929000


{'eval_loss': 0.1865186095237732,
 'eval_f1': 0.9310757174308866,
 'eval_precision': 0.9317299191063144,
 'eval_recall': 0.93175,
 'eval_accuracy': 0.93175,
 'eval_runtime': 4.7381,
 'eval_samples_per_second': 844.221,
 'eval_steps_per_second': 105.528,
 'epoch': 10.0}

In [11]:
trainer.save_model(f"{BASE_PATH}/models/transformerModels/distilBertModel")

### 2. BERT

In [12]:
from transformers import BertForSequenceClassification, BertTokenizer

In [13]:
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels = 6)
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
summary(model)

Layer (type:depth-idx)                                       Param #
BertForSequenceClassification                                --
├─BertModel: 1-1                                             --
│    └─BertEmbeddings: 2-1                                   --
│    │    └─Embedding: 3-1                                   23,440,896
│    │    └─Embedding: 3-2                                   393,216
│    │    └─Embedding: 3-3                                   1,536
│    │    └─LayerNorm: 3-4                                   1,536
│    │    └─Dropout: 3-5                                     --
│    └─BertEncoder: 2-2                                      --
│    │    └─ModuleList: 3-6                                  85,054,464
│    └─BertPooler: 2-3                                       --
│    │    └─Linear: 3-7                                      590,592
│    │    └─Tanh: 3-8                                        --
├─Dropout: 1-2                                               --
├─L

In [15]:
train_dataset = Dataset.from_dict({"text": train_data.tolist(),"labels" : train_label.tolist()})
test_dataset = Dataset.from_dict({"text": test_data.tolist(),"labels" : test_label.tolist()})

# Gabungkan menjadi DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [16]:
def tokenization(batch) : 
    return tokenizer(batch["text"], max_length = 100, padding = 'max_length', truncation = True, return_attention_mask = True)

dataset = dataset.map(tokenization, batch_size=32, batched = True)

Map: 100%|██████████| 4000/4000 [00:02<00:00, 1588.34 examples/s]


In [17]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def comp_metrics(acc) : 
    logits, labels = acc
    logits = np.argmax(logits, axis = -1)
    accuracy = accuracy_score(logits, labels)
    precision = precision_score(logits, labels, average='weighted')
    recall = recall_score(logits, labels, average='weighted')   
    f1 = f1_score(logits, labels, average='weighted')
    return {"f1" : f1, "precision" : precision, "recall" : recall, "accuracy" : accuracy}

trainingArgs = TrainingArguments(
    output_dir = "./bert_tuned",
    per_device_train_batch_size = 16,
    num_train_epochs = 10,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    fp16 = True,
    eval_strategy = "epoch",
    per_device_eval_batch_size = 8,
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "eval_f1",
    greater_is_better = True,
    seed = 42
)
trainer = Trainer(
    args = trainingArgs,
    model = model,
    train_dataset= dataset["train"],
    eval_dataset = dataset["test"],
    compute_metrics=comp_metrics
)

In [18]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.231200,0.196580,0.930621,0.932099,0.931000,0.931000
2,0.135800,0.158240,0.935688,0.936153,0.936250,0.936250
3,0.105100,0.202640,0.934690,0.939318,0.933000,0.933000
4,0.091600,0.228241,0.931749,0.932083,0.932250,0.932250
5,0.069400,0.315462,0.929311,0.929668,0.930000,0.930000
6,0.040500,0.367317,0.928115,0.929206,0.928500,0.928500
7,0.033500,0.369390,0.926735,0.926643,0.927000,0.927000
8,0.020800,0.396299,0.928547,0.928501,0.928750,0.928750
9,0.014300,0.431813,0.926147,0.926870,0.926500,0.926500
10,0.013700,0.417576,0.931253,0.931156,0.931500,0.931500


{'eval_loss': 0.15824024379253387,
 'eval_f1': 0.9356883091198221,
 'eval_precision': 0.9361533466147987,
 'eval_recall': 0.93625,
 'eval_accuracy': 0.93625,
 'eval_runtime': 7.9251,
 'eval_samples_per_second': 504.728,
 'eval_steps_per_second': 63.091,
 'epoch': 10.0}

In [19]:
trainer.save_model(f"{BASE_PATH}/models/transformerModels/bertModel")

### 3. RoBERTa

In [20]:
from transformers import RobertaForSequenceClassification, RobertaTokenizer

In [21]:
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels = 6)
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
summary(model)

Layer (type:depth-idx)                                            Param #
RobertaForSequenceClassification                                  --
├─RobertaModel: 1-1                                               --
│    └─RobertaEmbeddings: 2-1                                     --
│    │    └─Embedding: 3-1                                        38,603,520
│    │    └─Embedding: 3-2                                        394,752
│    │    └─Embedding: 3-3                                        768
│    │    └─LayerNorm: 3-4                                        1,536
│    │    └─Dropout: 3-5                                          --
│    └─RobertaEncoder: 2-2                                        --
│    │    └─ModuleList: 3-6                                       85,054,464
├─RobertaClassificationHead: 1-2                                  --
│    └─Linear: 2-3                                                590,592
│    └─Dropout: 2-4                                               --

In [23]:
train_dataset = Dataset.from_dict({"text": train_data.tolist(),"labels" : train_label.tolist()})
test_dataset = Dataset.from_dict({"text": test_data.tolist(),"labels" : test_label.tolist()})

# Gabungkan menjadi DatasetDict
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

In [24]:
def tokenization(batch) : 
    return tokenizer(batch["text"], max_length = 100, padding = 'max_length', truncation = True, return_attention_mask = True)

dataset = dataset.map(tokenization, batch_size=32, batched = True)

Map: 100%|██████████| 4000/4000 [00:01<00:00, 2534.75 examples/s]


In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def comp_metrics(acc) : 
    logits, labels = acc
    logits = np.argmax(logits, axis = -1)
    accuracy = accuracy_score(logits, labels)
    precision = precision_score(logits, labels, average='weighted')
    recall = recall_score(logits, labels, average='weighted')   
    f1 = f1_score(logits, labels, average='weighted')
    return {"f1" : f1, "precision" : precision, "recall" : recall, "accuracy" : accuracy}

trainingArgs = TrainingArguments(
    output_dir = "./roberta_tuned",
    per_device_train_batch_size = 16,
    num_train_epochs = 10,
    learning_rate = 2e-5,
    weight_decay = 0.01,
    fp16 = True,
    eval_strategy = "epoch",
    per_device_eval_batch_size = 8,
    save_strategy = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model = "eval_f1",
    greater_is_better = True,
    seed = 42
)
trainer = Trainer(
    args = trainingArgs,
    model = model,
    train_dataset= dataset["train"],
    eval_dataset = dataset["test"],
    compute_metrics=comp_metrics
)

In [26]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
1,0.307500,0.247168,0.917952,0.920605,0.917250,0.917250
2,0.184700,0.217249,0.928146,0.932516,0.929500,0.929500
3,0.146500,0.219178,0.932950,0.936594,0.932000,0.932000
4,0.133700,0.181237,0.929336,0.932220,0.930500,0.930500
5,0.104300,0.198648,0.935771,0.936833,0.936500,0.936500
6,0.095200,0.264160,0.930180,0.931929,0.931250,0.931250
7,0.077500,0.275806,0.929241,0.929837,0.930000,0.930000
8,0.066400,0.336697,0.929074,0.929290,0.929250,0.929250
9,0.051800,0.371155,0.926331,0.926504,0.926750,0.926750
10,0.036200,0.383091,0.927766,0.927607,0.928250,0.928250


{'eval_loss': 0.1986478716135025,
 'eval_f1': 0.9357705153774725,
 'eval_precision': 0.9368332013150706,
 'eval_recall': 0.9365,
 'eval_accuracy': 0.9365,
 'eval_runtime': 6.1393,
 'eval_samples_per_second': 651.542,
 'eval_steps_per_second': 81.443,
 'epoch': 10.0}

In [27]:
trainer.save_model(f"{BASE_PATH}/models/transformerModels/RoBertaModel")

### Evaluate the Model

In [5]:
tokenizerBert = BertTokenizer.from_pretrained("bert-base-uncased") 
modelBert = BertForSequenceClassification.from_pretrained(f"{BASE_PATH}/models/transformerModels/bertModel")
pipeBert = pipeline(task = "text-classification", model = modelBert, tokenizer = tokenizerBert)

tokenizerDistilBert = DistilBertTokenizer.from_pretrained("distilbert-base-uncased") 
modelDistilBert = DistilBertForSequenceClassification.from_pretrained(f"{BASE_PATH}/models/transformerModels/distilBertModel")
pipeDistilBert = pipeline(task = "text-classification", model = modelDistilBert, tokenizer = tokenizerDistilBert)

tokenizerRoBerta = RobertaTokenizer.from_pretrained("roberta-base") 
modelRoBerta = RobertaForSequenceClassification.from_pretrained(f"{BASE_PATH}/models/transformerModels/RoBertaModel")
pipeRoBerta = pipeline(task = "text-classification", model = modelRoBerta, tokenizer = tokenizerRoBerta)

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


In [ ]:
predictedBert = pipeBert(test_data.values.tolist())
predictedBert = [int(ans["label"].split("_")[-1]) for ans in predictedBert]

In [12]:
predictedDistilBert = pipeDistilBert(test_data.values.tolist())
predictedDistilBert = [int(ans["label"].split("_")[-1]) for ans in predictedDistilBert]

In [13]:
predictedRoBerta = pipeRoBerta(test_data.values.tolist())
predictedRoBerta = [int(ans["label"].split("_")[-1]) for ans in predictedRoBerta]

In [19]:
print("BERT PREDICTION RESULT : ")
print(classification_report(y_true = test_label, y_pred = predictedBert, digits = 3))

BERT PREDICTION RESULT : 
              precision    recall  f1-score   support

           0      0.971     0.970     0.971      1159
           1      0.969     0.932     0.950      1352
           2      0.814     0.909     0.859       328
           3      0.915     0.958     0.936       542
           4      0.927     0.888     0.908       475
           5      0.787     0.847     0.816       144

    accuracy                          0.936      4000
   macro avg      0.897     0.917     0.907      4000
weighted avg      0.938     0.936     0.937      4000



In [20]:
print("DISTILBERT PREDICTION RESULT : ")
print(classification_report(y_true = test_label, y_pred = predictedDistilBert, digits = 3))

DISTILBERT PREDICTION RESULT : 
              precision    recall  f1-score   support

           0      0.982     0.959     0.971      1159
           1      0.962     0.928     0.944      1352
           2      0.811     0.890     0.849       328
           3      0.890     0.970     0.929       542
           4      0.921     0.884     0.902       475
           5      0.783     0.854     0.817       144

    accuracy                          0.932      4000
   macro avg      0.892     0.914     0.902      4000
weighted avg      0.934     0.932     0.932      4000



In [21]:
print("ROBERTA PREDICTION RESULT : ")
print(classification_report(y_true = test_label, y_pred = predictedRoBerta, digits = 3))

ROBERTA PREDICTION RESULT : 
              precision    recall  f1-score   support

           0      0.984     0.959     0.971      1159
           1      0.953     0.946     0.950      1352
           2      0.833     0.884     0.858       328
           3      0.925     0.961     0.943       542
           4      0.945     0.865     0.903       475
           5      0.732     0.931     0.820       144

    accuracy                          0.936      4000
   macro avg      0.895     0.924     0.907      4000
weighted avg      0.940     0.936     0.937      4000

